[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/02-PDF_reporte.ipynb)


# 02 — Texto desde un PDF y lectura como en `01_Texto_y_maquina`

Aquí el documento ya es **PDF con texto digital** (no fotos escaneadas): podemos extraer caracteres con **`pypdf`** (el proyecto actual que continúa a **PyPDF2**). Eso **no** sustituye al OCR del notebook `03`: si el PDF fuera solo imagen, habría que usar Tesseract, EasyOCR, etc.

## Qué documento usamos

**EY — *100 casos rentables de IA* (2026)** (PDF en español): informe de negocio sobre casos de uso de inteligencia artificial, con vocabulario **sectorial** (empresa, datos, modelos, valor, riesgos…) además de las palabras muy frecuentes del español. Es **sustancioso** para practicar lo mismo que en **`01_Texto_y_maquina`**: tokens, limpieza con regex, vocabulario y frecuencias.

Los datos del módulo viven en la carpeta **`data/`** de este directorio. El notebook espera el archivo:

`data/ey_100_casos_rentables_ia_2026.pdf`

Si usas otro PDF, cambia la ruta en la celda siguiente o renombra el archivo.

---

## Instalación

In [ ]:
# %pip install -q pypdf wordcloud matplotlib

## Ubicar el PDF en `data/`

In [ ]:
from pathlib import Path

# Ruta al PDF (relativa a la carpeta `Modulo 4: NLP` — ahí debe estar `data/`).
PDF_LOCAL = Path("data/ey_100_casos_rentables_ia_2026.pdf")

print(PDF_LOCAL.resolve())

## Extraer todo el texto con `pypdf`

In [ ]:
from pypdf import PdfReader

lector = PdfReader(str(PDF_LOCAL))
num_paginas = len(lector.pages)
print(f"Páginas: {num_paginas}")

fragmentos: list[str] = []
for i, pagina in enumerate(lector.pages):
    t = pagina.extract_text()
    if t:
        fragmentos.append(t)

texto_completo = "\n".join(fragmentos)
print(f"Caracteres extraídos (aprox.): {len(texto_completo):,}")
print("\n--- Muestra (primeros 1200 caracteres) ---\n")
print(texto_completo[:1200])

## Limpieza y tokens (misma lógica que en `01_Texto_y_maquina`)

Pasamos a minúsculas, quitamos signos de puntuación frecuentes y partimos por espacios. En informes corporativos el PDF a veces inserta saltos raros; para un primer análisis basta esto.

In [ ]:
import re

texto_limpio = texto_completo.lower()
texto_limpio = re.sub(r"[.,;:!?¿¡'\"()\[\]{}—–-]", " ", texto_limpio)
texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

palabras = texto_limpio.split()
print(f"Total de tokens (aprox.): {len(palabras):,}")
print(f"Palabras únicas (vocabulario aprox.): {len(set(palabras)):,}")

## Qué palabras se repiten más

Las más frecuentes suelen ser **funcionales** (artículos, preposiciones). Más abajo filtramos una lista corta de **stopwords** en español solo para ilustrar otras palabras más “de contenido”.

In [ ]:
from collections import Counter

frecuencias = Counter(palabras)
print("Las 25 más frecuentes:\n")
for palabra, n in frecuencias.most_common(25):
    print(f"  {n:>6}  {palabra}")

In [ ]:
# Stopwords mínimas (solo didácticas; una librería como NLTK trae listas más completas)
STOP = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "y", "o", "u", "de", "del", "al",
    "en", "que", "con", "por", "para", "como", "se", "su", "sus", "lo", "le", "les", "a",
    "no", "si", "ya", "más", "menos", "tan", "entre", "sobre", "ser", "es", "son", "fue",
    "ha", "han", "he", "hay", "está", "están", "este", "esta", "esto", "ese", "esa", "eso",
}

palabras_filtradas = [p for p in palabras if p not in STOP and len(p) > 2]
frec_filtrada = Counter(palabras_filtradas)
print("25 más frecuentes tras quitar algunas stopwords:\n")
for palabra, n in frec_filtrada.most_common(25):
    print(f"  {n:>5}  {palabra}")

## Nube de palabras

Visualizamos las frecuencias ya filtradas: cuanto más grande la palabra, más veces aparece en el texto del PDF.

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud


def plot_wordcloud(contador, max_palabras=80):
    """Dibuja una nube a partir de un Counter (palabra → cuántas veces sale)."""
    frecuencias = dict(contador.most_common(500))
    nube = WordCloud(
        width=900,
        height=500,
        background_color="white",
        max_words=max_palabras,
        colormap="viridis",
    ).generate_from_frequencies(frecuencias)

    plt.figure(figsize=(11, 6))
    plt.imshow(nube, interpolation="bilinear")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


plot_wordcloud(frec_filtrada)

## Cierre

- **`pypdf`** lee la capa de texto del PDF; si el archivo fuera un escaneo sin OCR, obtendrías casi vacío y habría que pasar al flujo del notebook **03**.
- El análisis de frecuencias conecta directamente con **vocabulario, repetición y límites del tokenizado “por espacios”** del notebook **`01_Texto_y_maquina`**.

**Otras ideas** (misma técnica, otro PDF con texto digital en `data/`): informes de **CEPAL**, **INEGI**, **Banco de México**, **ONU/PNUD**, u otros documentos de negocio o sector público — siempre revisa **licencia** y uso educativo.